# HDF5 vector/scalar inspector

Inspect raw datasets in an analysis HDF5 file to debug NaNs after DataFrame extraction.

Run the cell below after setting `H5_PATH` and `COLUMNS` as needed.


In [6]:
import hdf5plugin
from pathlib import Path
import h5py
import numpy as np
from src.utils.io_functions import resolve_h5_inputs
ROOT = Path.cwd().parent.parent

H5_SPEC = "outputs/test_parametric_T_seeded"  # folder, file path, or "latest"
COLUMNS = ["I_target", "P_DT_eq", "P_aux", "t_startup", "unrealized_profits"]
FILTER_SUCCESS = True
MAX_PRINT = 25

# ------------------------

# Resolve HDF5 file (supports folder/latest spec)
resolved, _ = resolve_h5_inputs(H5_SPEC, root=ROOT)
if not resolved:
    raise FileNotFoundError(f"No .h5 found for spec: {H5_SPEC}")
h5_path = resolved[0]
print(f"Inspecting: {h5_path}")

with h5py.File(h5_path, "r") as h5:
    all_keys = list(h5.keys())
    print(f"Datasets found ({len(all_keys)}): {all_keys}")
    targets = list(COLUMNS) if COLUMNS else all_keys
    mask = None
    if FILTER_SUCCESS and "sol_success" in h5:
        mask = np.asarray(h5["sol_success"][...]).astype(bool)
        print(f"sol_success present: {mask.sum()} / {len(mask)} successful")

    for col in targets:
        if col not in h5:
            print(f"\nColumn '{col}' not found.")
            continue
        data = h5[col][...]
        print(f"\nColumn: {col}")
        print(f"  shape: {data.shape}")
        print(f"  dtype: {data.dtype}")
        if data.ndim == 1 and mask is not None and len(mask) == len(data):
            data = data[mask]
            print(f"  filtered to sol_success=True -> {data.shape[0]} rows")
        else:
            print("  not filtered (mask length mismatch or vector data)")
        flat = np.ravel(data)
        shown = flat[:MAX_PRINT]
        print(f"  sample values (first {len(shown)}): {shown}")
        if flat.size > len(shown):
            print(f"  ... ({flat.size - len(shown)} more)")


Inspecting: /home/alessmor/Scrivania/dd_startup/outputs/test_parametric_T_seeded/ddstartup_20251113_002935_parametric_T_seeded.h5
Datasets found (35): ['E_lost', 'I_target', 'N_ifc', 'N_ofc', 'N_stor', 'P_DDn', 'P_DDp', 'P_DT', 'P_DT_eq', 'P_aux', 'P_aux_DT_eq', 'Q_DD', 'Q_DT_eq', 'TBE', 'TBR_DDn', 'TBR_DT', 'T_i', 'V_plasma', 'capacity_factor', 'error', 'eta_th', 'linear_index', 'n_D', 'n_He3', 'n_T', 'n_tot', 'parameter_fields', 'price_of_electricity', 'sol_success', 't_startup', 'tau_ifc', 'tau_ofc', 'tau_p_He3', 'tau_p_T', 'unrealized_profits']
sol_success present: 5943770 / 6237000 successful

Column: I_target
  shape: (6237000,)
  dtype: float64
  filtered to sol_success=True -> 5943770 rows
  sample values (first 25): [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan]
  ... (5943745 more)

Column: P_DT_eq
  shape: (6237000,)
  dtype: float64
  filtered to sol_success=True -> 5943770 rows
  sample values (first 25): [39242642.39

In [11]:
# Scan a metric for rows where vector/scalar values match a target or fall in a range.
# - METRIC: dataset name
# - TARGET_VALUE: value to match (use np.nan for NaN); set to None to skip exact match
# - VALUE_RANGE: (min, max) inclusive; use None for open bounds or to disable
# - POS_RANGE: (start, end) positions inside the vector to check; None means all positions
# - MAX_ROWS: stop after reporting this many rows; None processes all rows

METRIC = 'TBE'
TARGET_VALUE = np.nan  # use np.nan for NaNs; set to None to ignore exact match
VALUE_RANGE = (None, None)  # e.g., (0.0, 1.0); set both to None to ignore
POS_RANGE = (-5, -1)  # e.g., (0, 4) to check first 5 positions; None checks all
MAX_ROWS = None  # None to scan all rows
CHUNK = 10000

with h5py.File(h5_path, 'r') as h5:
    if METRIC not in h5:
        print(f"Metric '{METRIC}' not found in H5.")
    else:
        ds = h5[METRIC]
        sol_mask = None
        if FILTER_SUCCESS and 'sol_success' in h5 and h5['sol_success'].shape[0] == ds.shape[0]:
            sol_mask = np.asarray(h5['sol_success'][...]).astype(bool)
        total = ds.shape[0]
        matches = []
        match_count = 0
        processed = 0
        pos_start, pos_end = POS_RANGE
        for start in range(0, total, CHUNK):
            end = min(total, start + CHUNK)
            data = ds[start:end]
            if sol_mask is not None:
                data = data[sol_mask[start:end]]
            arr = np.asarray(data)
            # Normalize to 2D: rows x elements
            if arr.ndim == 0:
                arr = arr.reshape(1, 1)
            elif arr.ndim == 1:
                arr = arr.reshape(-1, 1)
            elif arr.dtype == object:
                arr = np.array([np.asarray(v).ravel() for v in arr], dtype=object)
            # Determine positions to check
            rows = arr.shape[0]
            for i in range(rows):
                vec = arr[i]
                v = np.asarray(vec).ravel() if not isinstance(vec, np.ndarray) or vec.dtype != object else np.asarray(vec)
                if v.size == 0:
                    continue
                start_pos = 0 if pos_start is None else max(0, pos_start)
                end_pos = v.size if pos_end is None else min(v.size, pos_end + 1)
                slice_v = v[start_pos:end_pos]
                cond = np.ones_like(slice_v, dtype=bool)
                if TARGET_VALUE is not None:
                    if isinstance(TARGET_VALUE, float) and np.isnan(TARGET_VALUE):
                        cond &= np.isnan(slice_v)
                    else:
                        cond &= (slice_v == TARGET_VALUE)
                lo, hi = VALUE_RANGE
                if lo is not None:
                    cond &= slice_v >= lo
                if hi is not None:
                    cond &= slice_v <= hi
                bad_idx = np.nonzero(cond)[0]
                if bad_idx.size:
                    match_count += 1
                    if MAX_ROWS is None or len(matches) < MAX_ROWS:
                        orig_pos = [start_pos + int(x) for x in bad_idx]
                        pos_list = ",".join(str(p) for p in orig_pos[:10])
                        if len(orig_pos) > 10:
                            pos_list += f", ... (+{len(orig_pos) - 10})"
                        matches.append((start + i, pos_list, v.size))
                    if MAX_ROWS is not None and len(matches) >= MAX_ROWS:
                        processed += rows
                        break
            processed += rows
            if MAX_ROWS is not None and len(matches) >= MAX_ROWS:
                break
        print(f"{METRIC}: {match_count} row(s) matched after sol_success filter (processed {processed} rows)")
        for row_idx, pos_list, vlen in matches:
            print(f"  row {row_idx}: match at positions [{pos_list}] (len={vlen})")
        if MAX_ROWS is not None and match_count > MAX_ROWS:
            print(f"  ... first {MAX_ROWS} rows shown")


TBE: 0 row(s) matched after sol_success filter (processed 5943770 rows)
